# Practical Exam: House sales

RealAgents is a real estate company that focuses on selling houses.

RealAgents sells a variety of types of house in one metropolitan area.

Some houses sell slowly and sometimes require lowering the price in order to find a buyer.

In order to stay competitive, RealAgents would like to optimize the listing prices of the houses it is trying to sell.

They want to do this by predicting the sale price of a house given its characteristics.

If they can predict the sale price in advance, they can decrease the time to sale.


## Data

The dataset contains records of previous houses sold in the area.

| Column Name | Criteria                                                |
|-------------|---------------------------------------------------------|
| house_id    | Nominal. </br> Unique identifier for houses. </br>Missing values not possible. |
| city        | Nominal. </br>The city in which the house is located. One of 'Silvertown', 'Riverford', 'Teasdale' and 'Poppleton'. </br>Replace missing values with "Unknown". |
| sale_price  | Discrete. </br>The sale price of the house in whole dollars. Values can be any positive number greater than or equal to zero.</br>Remove missing entries. |
| sale_date   | Discrete. </br>The date of the last sale of the house. </br>Replace missing values with 2023-01-01. |
| months_listed  | Continuous. </br>The number of months the house was listed on the market prior to its last sale, rounded to one decimal place. </br>Replace missing values with mean number of months listed, to one decimal place. |
| bedrooms    | Discrete. </br>The number of bedrooms in the house. Any positive values greater than or equal to zero. </br>Replace missing values with the mean number of bedrooms, rounded to the nearest integer. |
| house_type   | Ordinal. </br>One of "Terraced" (two shared walls), "Semi-detached" (one shared wall), or "Detached" (no shared walls). </br>Replace missing values with the most common house type. |
| area      | Continuous. </br>The area of the house in square meters, rounded to one decimal place. </br>Replace missing values with the mean, to one decimal place. |


In [22]:
import pandas as pd 
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error as MSE 

# Task 1

The team at RealAgents knows that the city that a property is located in makes a difference to the sale price. 

Unfortuntately they believe that this isn't always recorded in the data. 

Calculate the number of missing values of the `city`. 

 - You should use the data in the file "house_sales.csv". 

 - Your output should be an object `missing_city`, that contains the number of missing values in this column. 

In [11]:
df_house = pd.read_csv('./data/house_sales.csv')
missing_city = df_house['city'].value_counts().iloc[-1]
print(f'In the column city are {missing_city} missing values')
df_house.head()

In the column city are 73 missing values


,house_id,city,sale_price,sale_date,months_listed,bedrooms,house_type,area
0,1217792,Silvertown,55943,2021-09-12,5.4,2,Semi-detached,107.8 sq.m.
1,1900913,Silvertown,384677,2021-01-17,6.3,5,Detached,498.8 sq.m.
2,1174927,Riverford,281707,2021-11-10,6.9,6,Detached,542.5 sq.m.
3,1773666,Silvertown,373251,2020-04-13,6.1,6,Det.,528.4 sq.m.
4,1258487,Silvertown,328885,2020-09-24,8.7,5,Detached,477.1 sq.m.


# Task 2 

Before you fit any models, you will need to make sure the data is clean. 

The table below shows what the data should look like. 

Create a cleaned version of the dataframe. 

 - You should start with the data in the file "house_sales.csv". 

 - Your output should be a dataframe named `clean_data`. 

 - All column names and values should match the table below.


| Column Name | Criteria                                                |
|-------------|---------------------------------------------------------|
| house_id    | Nominal. </br> Unique identifier for houses. </br>Missing values not possible. |
| city        | Nominal. </br>The city in which the house is located. One of 'Silvertown', 'Riverford', 'Teasdale' and 'Poppleton' </br>Replace missing values with "Unknown". |
| sale_price  | Discrete. </br>The sale price of the house in whole dollars. Values can be any positive number greater than or equal to zero.</br>Remove missing entries. |
| sale_date   | Discrete. </br>The date of the last sale of the house. </br>Replace missing values with 2023-01-01. |
| months_listed  | Continuous. </br>The number of months the house was listed on the market prior to its last sale, rounded to one decimal place. </br>Replace missing values with mean number of months listed, to one decimal place. |
| bedrooms    | Discrete. </br>The number of bedrooms in the house. Any positive values greater than or equal to zero. </br>Replace missing values with the mean number of bedrooms, rounded to the nearest integer. |
| house_type   | Ordinal. </br>One of "Terraced", "Semi-detached", or "Detached". </br>Replace missing values with the most common house type. |
| area      | Continuous. </br>The area of the house in square meters, rounded to one decimal place. </br>Replace missing values with the mean, to one decimal place. |

In [19]:
print("----- Before the preprocessing -----")
print(f"'city' column: {df_house['city'].unique()} with type: {type(df_house['city'][0])}")
print(f"'sale_price' column: {df_house['sale_price'].isna().sum()} missing values and with type: {type(df_house['sale_price'][0])}")
print(f"'sale_date' column: {df_house['sale_date'].isna().sum()} missing values and with type: {type(df_house['sale_date'][0])}")
print(f"'months_listed' column: {df_house['months_listed'].isna().sum()} missing values and with type: {type(df_house['months_listed'][0])}")
print(f"'bedrooms' column: {df_house['bedrooms'].unique()} with type: {type(df_house['bedrooms'][0])}")
print(f"'house_type' column: {df_house['house_type'].unique()} with type: {type(df_house['house_type'][0])}")
print(f"'area' column: {df_house['area'].isna().sum()} missing values and with type: {type(df_house['area'][0])}")


----- Before the preprocessing -----
'city' column: ['Silvertown' 'Riverford' 'Teasdale' 'Poppleton' '--'] with type: <class 'str'>
'sale_price' column: 0 missing values and with type: <class 'numpy.int64'>
'sale_date' column: 0 missing values and with type: <class 'str'>
'months_listed' column: 31 missing values and with type: <class 'numpy.float64'>
'bedrooms' column: [2 5 6 4 3] with type: <class 'numpy.int64'>
'house_type' column: ['Semi-detached' 'Detached' 'Det.' 'Terraced' 'Semi' 'Terr.'] with type: <class 'str'>
'area' column: 0 missing values and with type: <class 'str'>


In [20]:
# city: replace '--' with Unkonw
city_values = ['Silvertown', 'Riverford', 'Teasdale', 'Poppleton']
df_house['city'] = df_house['city'].apply(lambda x: x if x in city_values else 'Unknown')
# sale_price: nothing to do
# sale_date: nothing to do -> type of str is ok
# month_listed: replace missing values with mean
df_house['months_listed'] = df_house['months_listed'].fillna(df_house['months_listed'].mean())
df_house['months_listed'] = df_house['months_listed'].apply(lambda x: f"{x:.1f}").astype(float)
# bedrooms: nothing to do
# house_type: replace ['Det.', 'Semi', 'Terr.']
house_type_mapping = {
    'Det.': 'Detached',
    'Semi': 'Semi-detached',
    'Terr.': 'Terraced'
}
df_house['house_type'] = df_house['house_type'].replace(house_type_mapping)
# area: remove str from value and convert to float
df_house['area'] = (
    df_house['area']
    .str.replace(r'[^0-9.]', '', regex=True)              
    .str.replace(r'(\.\d*)\.+', r'\1', regex=True)    
)
df_house['area'] = df_house['area'].astype(float)

# copy dataframe for Datacamp evaluation
clean_data = df_house.copy()
clean_data.head()

,house_id,city,sale_price,sale_date,months_listed,bedrooms,house_type,area
0,1217792,Silvertown,55943,2021-09-12,5.4,2,Semi-detached,107.8
1,1900913,Silvertown,384677,2021-01-17,6.3,5,Detached,498.8
2,1174927,Riverford,281707,2021-11-10,6.9,6,Detached,542.5
3,1773666,Silvertown,373251,2020-04-13,6.1,6,Detached,528.4
4,1258487,Silvertown,328885,2020-09-24,8.7,5,Detached,477.1


# Task 3 

The team at RealAgents have told you that they have always believed that the number of bedrooms is the biggest driver of house price. 

Producing a table showing the difference in the average sale price by number of bedrooms along with the variance to investigate this question for the team.

 - You should start with the data in the file 'house_sales.csv'.

 - Your output should be a data frame named `price_by_rooms`. 

 - It should include the three columns `bedrooms`, `avg_price`, `var_price`. 

 - Your answers should be rounded to 1 decimal place.   

In [21]:
avg_price = df_house.groupby(['bedrooms'])['sale_price'].mean().round(1)
var_price = df_house.groupby(['bedrooms'])['sale_price'].var().round(1)


price_by_rooms = pd.DataFrame({
    'bedrooms': avg_price.index,
    'avg_price': avg_price.values,
    'var_price': var_price.values
})

print(price_by_rooms)

   bedrooms  avg_price     var_price
0         2    67076.4  5.652896e+08
1         3   154665.1  2.378289e+09
2         4   234704.6  1.725211e+09
3         5   301515.9  2.484328e+09
4         6   375741.3  3.924432e+09


# Task 4

Fit a baseline model to predict the sale price of a house.

 1. Fit your model using the data contained in “train.csv” </br></br>

 2. Use “validation.csv” to predict new values based on your model. You must return a dataframe named `base_result`, that includes `house_id` and `price`. The price column must be your predicted values.

In [39]:
df_train = pd.read_csv('./data/train.csv')
df_train = pd.get_dummies(df_train, columns=['city', 'house_type'], drop_first=True)
X_train = df_train.drop(['house_id', 'sale_price', 'sale_date'] , axis=1)
y_train = df_train['sale_price']

df_val = pd.read_csv('./data/validation.csv')
df_val = pd.get_dummies(df_val, columns=['city', 'house_type'], drop_first=True)

X_val = df_val.drop(['house_id', 'sale_date'] , axis=1)
X_val.head()

,months_listed,bedrooms,area,city_Riverford,city_Silvertown,city_Teasdale,house_type_Semi-detached,house_type_Terraced
0,7.7,3,209.7,False,False,True,False,True
1,6.5,4,390.6,False,False,True,False,False
2,7.4,6,556.8,False,True,False,False,False
3,8.8,3,208.3,False,True,False,True,False
4,5.7,4,389.2,False,True,False,False,False


In [25]:
rf = RandomForestRegressor(random_state=42) 
params_rf = { 
              'n_estimators': [100, 200, 300, 400], 
              'max_depth': [2, 4, 6], 
              'min_samples_leaf': [0.1, 0.2], 
              'max_features': ['log2', 'sqrt'] 
             } 
grid_rf = GridSearchCV(estimator=rf, 
                       param_grid=params_rf,  
                       cv=3, 
                       scoring='neg_mean_squared_error', 
                       verbose=1, 
                       n_jobs=-1)
grid_rf.fit(X_train, y_train) 

Fitting 3 folds for each of 48 candidates, totalling 144 fits


GridSearchCV(cv=3, estimator=RandomForestRegressor(random_state=42), n_jobs=-1,
             param_grid={'max_depth': [2, 4, 6],
                         'max_features': ['log2', 'sqrt'],
                         'min_samples_leaf': [0.1, 0.2],
                         'n_estimators': [100, 200, 300, 400]},
             scoring='neg_mean_squared_error', verbose=1)

In [30]:
best_hyperparams_rf = grid_rf.best_params_ 
best_model_rf = grid_rf.best_estimator_ 
print('Best hyperparameters:\n', best_hyperparams_rf) 

y_predict_train = best_model_rf.predict(X_train) 
RMSE = MSE(y_train, y_predict_train) ** 1/2
print('Train RMSE: {:.2f}'.format(RMSE)) 

Best hyperparameters:
 {'max_depth': 4, 'max_features': 'log2', 'min_samples_leaf': 0.1, 'n_estimators': 300}
Train RMSE: 1157771287.72


In [31]:
base_result = pd.DataFrame()
base_result = df_val[['house_id']].copy()
base_result['price'] = best_model_rf.predict(X_val)
print(base_result)

     house_id          price
0     1331375  152719.319159
1     1630115  263164.846430
2     1645745  338162.715280
3     1336775  111516.699520
4     1888274  266753.038911
..        ...            ...
295   1986255  333964.401249
296   1896276  337535.387087
297   1758223  260017.584359
298   1752010  156295.181563
299   1651404  334380.198193

[300 rows x 2 columns]


# Task 5

Fit a comparison model to predict the sale price of a house.

 1. Fit your model using the data contained in “train.csv” </br></br>

 2. Use “validation.csv” to predict new values based on your model. You must return a dataframe named `compare_result`, that includes `house_id` and `price`. The price column must be your predicted values.

In [29]:
df_train = pd.read_csv('./data/train.csv')
df_train = pd.get_dummies(df_train, columns=['city', 'house_type'], drop_first=True)
X_train = df_train.drop(['house_id', 'sale_price', 'sale_date'] , axis=1)
y_train = df_train['sale_price']

df_val = pd.read_csv('./data/validation.csv')
df_val = pd.get_dummies(df_val, columns=['city', 'house_type'], drop_first=True)

X_val = df_val.drop(['house_id', 'sale_date'] , axis=1)
X_val.head()

,months_listed,bedrooms,area,city_Riverford,city_Silvertown,city_Teasdale,house_type_Semi-detached,house_type_Terraced
0,7.7,3,209.7,False,False,True,False,True
1,6.5,4,390.6,False,False,True,False,False
2,7.4,6,556.8,False,True,False,False,False
3,8.8,3,208.3,False,True,False,True,False
4,5.7,4,389.2,False,True,False,False,False


In [ ]:
gbt = GradientBoostingRegressor(random_state=42)
params_gbt = { 
              'n_estimators': [100, 200, 300, 400], 
              'max_depth': [1, 2, 3, 4],

             } 

grid_gbt = GridSearchCV(estimator=gbt, 
                       param_grid=params_gbt,  
                       cv=3, 
                       scoring='neg_mean_squared_error', 
                       verbose=1, 
                       n_jobs=-1)

grid_gbt.fit(X_train, y_train) 

Fitting 3 folds for each of 16 candidates, totalling 48 fits


GridSearchCV(cv=3, estimator=GradientBoostingRegressor(random_state=42),
             n_jobs=-1,
             param_grid={'max_depth': [1, 2, 3, 4],
                         'n_estimators': [100, 200, 300, 400]},
             scoring='neg_mean_squared_error', verbose=1)

In [34]:
best_hyperparams_gbt = grid_gbt.best_params_ 
best_model_gbt = grid_gbt.best_estimator_ 
print('Best hyperparameters:\n', best_hyperparams_gbt) 

y_predict_train = best_model_gbt.predict(X_train) 
RMSE = MSE(y_train, y_predict_train) ** 1/2
print('Train RMSE: {:.2f}'.format(RMSE)) 

Best hyperparameters:
 {'max_depth': 4, 'n_estimators': 100}
Train RMSE: 53425232.90


In [35]:
compare_result = pd.DataFrame()
compare_result = df_val[['house_id']].copy()
compare_result['price'] = best_model_gbt.predict(X_val)
print(compare_result)

     house_id          price
0     1331375   81938.359525
1     1630115  304015.647176
2     1645745  402324.325367
3     1336775  112841.943018
4     1888274  267907.119954
..        ...            ...
295   1986255  356139.664804
296   1896276  382152.426004
297   1758223  260177.832746
298   1752010  175817.750276
299   1651404  418890.931672

[300 rows x 2 columns]
